# BLVCKDISK AI Studio — one real video (Kaggle, free GPU)

Generates **one** short clip (text-to-video or image-to-video) with **Wan 2.2 TI2V 5B** (Apache-2.0) in ComfyUI, on Kaggle's free GPU.
Settings prepared on blackdisk.co/studio are already filled in at step 5.

**Before running:** right sidebar → *Session options* → **Accelerator: GPU T4 x2** and **Internet: On**.
For image mode, upload your image (step 5). Then use **Run All**. The first run downloads ~17 GB of model files.
Nothing here costs money. When finished, click **Stop session** (power icon) to save your weekly GPU hours.

## 1 · Check the machine

In [ ]:
import shutil, subprocess, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "NO GPU — set Accelerator to GPU T4 x2 in Session options, then restart.")
print(f"RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3:.1f} GB")
WORK = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
os.makedirs(WORK, exist_ok=True)
free = shutil.disk_usage(WORK).free / 1024**3
print(f"Free disk in {WORK}: {free:.0f} GB")
assert free > 25, "Need ~25 GB free for ComfyUI + models."

## 2 · Install ComfyUI (official repository)

In [ ]:
COMFY = f"{WORK}/ComfyUI"
if not os.path.isdir(COMFY):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}
# Keep Kaggle's preinstalled PyTorch (CUDA build); install the rest.
!grep -viE '^(torch|torchvision|torchaudio)([^a-z]|$)' {COMFY}/requirements.txt > /tmp/comfy-req.txt
!pip install -q -r /tmp/comfy-req.txt
!cd {COMFY} && git log -1 --format='ComfyUI commit %h (%cs)'

## 3 · Download the model (official ComfyUI repackage on Hugging Face, ~17 GB, Apache-2.0)

In [ ]:
FILES = {
    "diffusion_models/wan2.2_ti2v_5B_fp16.safetensors": "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
    "text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors": "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "vae/wan2.2_vae.safetensors": "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors",
}
for rel, url in FILES.items():
    dest = f"{COMFY}/models/{rel}"
    if os.path.exists(dest) and os.path.getsize(dest) > 1e8:
        print("already have", rel); continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print("downloading", rel)
    !wget -q --show-progress -O {dest}.part "{url}" && mv {dest}.part {dest}
!du -sh {COMFY}/models/diffusion_models {COMFY}/models/text_encoders {COMFY}/models/vae

## 4 · Studio workflow code (embedded copy of `studio-agent/comfy.py` and `workflows.py`)

In [ ]:
%%writefile /tmp/comfy.py
"""
Minimal ComfyUI client (standard library only).

Talks to ComfyUI on the same machine (default http://127.0.0.1:8188). ComfyUI itself is never
exposed to the internet; in the Kaggle notebook it listens on 127.0.0.1 only.
"""
from __future__ import annotations

import base64
import json
import os
import socket
import struct
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
import uuid


class ComfyError(Exception):
    pass


class ComfyClient:
    def __init__(self, base_url: str, timeout: float = 15.0):
        self.base = base_url.rstrip("/")
        self.timeout = timeout
        self.client_id = uuid.uuid4().hex
        # prompt_id -> {"value": int, "max": int, "node": str|None, "updated": float}
        self.progress: dict[str, dict] = {}
        self.executing: str | None = None
        self._lock = threading.Lock()

    # ---------------------------------------------------------------- HTTP
    def _request(self, method: str, path: str, body: bytes | None = None, headers: dict | None = None, timeout: float | None = None):
        req = urllib.request.Request(self.base + path, data=body, method=method, headers=headers or {})
        try:
            return urllib.request.urlopen(req, timeout=timeout or self.timeout)
        except urllib.error.HTTPError as e:
            detail = e.read()[:2000].decode("utf-8", "replace")
            raise ComfyError(f"ComfyUI {method} {path} → HTTP {e.code}: {detail}") from None
        except (urllib.error.URLError, OSError) as e:
            raise ComfyError(f"ComfyUI is not reachable at {self.base} ({e})") from None

    def get_json(self, path: str):
        with self._request("GET", path) as r:
            return json.loads(r.read())

    def post_json(self, path: str, data) -> dict:
        body = json.dumps(data).encode()
        with self._request("POST", path, body, {"Content-Type": "application/json"}) as r:
            raw = r.read()
            return json.loads(raw) if raw.strip() else {}

    # ---------------------------------------------------------------- API
    def system_stats(self) -> dict:
        return self.get_json("/system_stats")

    def object_info(self, node_class: str) -> dict:
        return self.get_json("/object_info/" + urllib.parse.quote(node_class)).get(node_class, {})

    def queue_prompt(self, graph: dict) -> str:
        res = self.post_json("/prompt", {"prompt": graph, "client_id": self.client_id})
        if res.get("node_errors"):
            raise ComfyError("ComfyUI rejected the workflow: " + json.dumps(res["node_errors"])[:1500])
        pid = res.get("prompt_id")
        if not pid:
            raise ComfyError("ComfyUI did not return a prompt_id: " + json.dumps(res)[:500])
        return pid

    def history(self, prompt_id: str) -> dict | None:
        return self.get_json("/history/" + urllib.parse.quote(prompt_id)).get(prompt_id)

    def queue(self) -> dict:
        return self.get_json("/queue")

    def interrupt(self, prompt_id: str) -> None:
        self.post_json("/interrupt", {"prompt_id": prompt_id})

    def delete_pending(self, prompt_id: str) -> None:
        self.post_json("/queue", {"delete": [prompt_id]})

    def upload_image(self, filename: str, data: bytes, content_type: str, subfolder: str = "studio") -> str:
        """Uploads to ComfyUI's input folder. Returns the value LoadImage expects ("subfolder/name")."""
        boundary = uuid.uuid4().hex
        parts = []
        for name, value in (("type", "input"), ("subfolder", subfolder), ("overwrite", "true")):
            parts.append(f'--{boundary}\r\nContent-Disposition: form-data; name="{name}"\r\n\r\n{value}\r\n'.encode())
        parts.append(
            f'--{boundary}\r\nContent-Disposition: form-data; name="image"; filename="{filename}"\r\nContent-Type: {content_type}\r\n\r\n'.encode()
            + data + b"\r\n"
        )
        parts.append(f"--{boundary}--\r\n".encode())
        with self._request("POST", "/upload/image", b"".join(parts), {"Content-Type": f"multipart/form-data; boundary={boundary}"}, timeout=60) as r:
            res = json.loads(r.read())
        return f"{res['subfolder']}/{res['name']}" if res.get("subfolder") else res["name"]

    def download_output(self, filename: str, subfolder: str, folder_type: str, dest: str) -> int:
        q = urllib.parse.urlencode({"filename": filename, "subfolder": subfolder, "type": folder_type})
        tmp = dest + ".part"
        size = 0
        with self._request("GET", "/view?" + q, timeout=300) as r, open(tmp, "wb") as f:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                size += len(chunk)
        os.replace(tmp, dest)
        return size

    # ---------------------------------------------------------------- live progress (websocket)
    def start_progress_listener(self) -> None:
        threading.Thread(target=self._ws_loop, name="comfy-ws", daemon=True).start()

    def progress_for(self, prompt_id: str) -> dict | None:
        with self._lock:
            p = self.progress.get(prompt_id)
            return dict(p) if p else None

    def _ws_loop(self) -> None:
        while True:
            try:
                self._ws_session()
            except Exception:
                pass  # ComfyUI offline or restarted — progress is optional; status still comes from /history.
            time.sleep(3)

    def _ws_session(self) -> None:
        u = urllib.parse.urlparse(self.base)
        host, port = u.hostname or "127.0.0.1", u.port or 80
        sock = socket.create_connection((host, port), timeout=10)
        try:
            key = base64.b64encode(os.urandom(16)).decode()
            sock.sendall(
                (f"GET /ws?clientId={self.client_id} HTTP/1.1\r\nHost: {host}:{port}\r\nUpgrade: websocket\r\n"
                 f"Connection: Upgrade\r\nSec-WebSocket-Key: {key}\r\nSec-WebSocket-Version: 13\r\n\r\n").encode()
            )
            head = b""
            while b"\r\n\r\n" not in head:
                chunk = sock.recv(1024)
                if not chunk:
                    return
                head += chunk
            if b" 101 " not in head.split(b"\r\n", 1)[0]:
                return
            buf = head.split(b"\r\n\r\n", 1)[1]
            sock.settimeout(60)
            while True:
                opcode, payload, buf = self._read_frame(sock, buf)
                if opcode == 1:
                    self._on_message(payload)
                elif opcode == 9:  # ping → pong (client frames must be masked)
                    self._send_frame(sock, 10, payload)
                elif opcode == 8:
                    return
                # opcode 2 = binary preview images — ignored
        finally:
            sock.close()

    @staticmethod
    def _recv_exact(sock, buf: bytes, n: int) -> tuple[bytes, bytes]:
        while len(buf) < n:
            chunk = sock.recv(65536)
            if not chunk:
                raise ConnectionError("websocket closed")
            buf += chunk
        return buf[:n], buf[n:]

    def _read_frame(self, sock, buf: bytes):
        hdr, buf = self._recv_exact(sock, buf, 2)
        opcode = hdr[0] & 0x0F
        length = hdr[1] & 0x7F
        if length == 126:
            ext, buf = self._recv_exact(sock, buf, 2)
            length = struct.unpack(">H", ext)[0]
        elif length == 127:
            ext, buf = self._recv_exact(sock, buf, 8)
            length = struct.unpack(">Q", ext)[0]
        mask = b""
        if hdr[1] & 0x80:
            mask, buf = self._recv_exact(sock, buf, 4)
        payload, buf = self._recv_exact(sock, buf, length)
        if mask:
            payload = bytes(b ^ mask[i % 4] for i, b in enumerate(payload))
        return opcode, payload, buf

    @staticmethod
    def _send_frame(sock, opcode: int, payload: bytes) -> None:
        mask = os.urandom(4)
        n = len(payload)
        head = bytes([0x80 | opcode])
        head += bytes([0x80 | n]) if n < 126 else bytes([0x80 | 126]) + struct.pack(">H", n)
        sock.sendall(head + mask + bytes(b ^ mask[i % 4] for i, b in enumerate(payload)))

    def _on_message(self, payload: bytes) -> None:
        try:
            msg = json.loads(payload)
        except ValueError:
            return
        data = msg.get("data") or {}
        pid = data.get("prompt_id")
        with self._lock:
            if msg.get("type") == "progress" and pid:
                self.progress[pid] = {"value": int(data.get("value", 0)), "max": int(data.get("max", 0)), "node": data.get("node"), "updated": time.time()}
            elif msg.get("type") == "executing":
                self.executing = pid if data.get("node") is not None else None
            if len(self.progress) > 200:
                for old in sorted(self.progress, key=lambda k: self.progress[k]["updated"])[:100]:
                    self.progress.pop(old, None)

In [ ]:
%%writefile /tmp/workflows.py
"""
Generation workflows the studio can run. Each workflow declares exactly what the model
supports; the website builds its controls from this, so it never offers a setting the model
cannot produce.

Wan 2.2 TI2V-5B
  - Weights: Apache-2.0 (Wan-AI/Wan2.2-TI2V-5B). Text encoder umt5-xxl: Apache-2.0.
  - Files (ComfyUI repackage, linked from ComfyUI's official template "video_wan2_2_5B_ti2v"):
      models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors   (9.31 GB)
      models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors (6.27 GB)
      models/vae/wan2.2_vae.safetensors                          (1.31 GB)
  - Native output: 1280×704 or 704×1280, 24 fps, up to 121 frames (≈5 s) per clip.
    Graph and defaults mirror the official template (20 steps, CFG 5, uni_pc/simple, shift 8).
"""
from __future__ import annotations

# Wan's recommended negative prompt (as shipped in the official ComfyUI template).
WAN_DEFAULT_NEGATIVE = (
    "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，"
    "多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
)

WAN22_FILES = {
    "diffusion_models": "wan2.2_ti2v_5B_fp16.safetensors",
    "text_encoders": "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "vae": "wan2.2_vae.safetensors",
}

# Every size is a multiple of 32 (the latent node's step). "native" = the resolution the model was trained for.
WAN22_SIZES = [
    {"id": "16x9", "aspect": "16:9", "width": 1280, "height": 704, "label": "16:9 · 1280×704 (native 720p)", "native": True},
    {"id": "9x16", "aspect": "9:16", "width": 704, "height": 1280, "label": "9:16 · 704×1280 (native 720p)", "native": True},
    {"id": "1x1", "aspect": "1:1", "width": 960, "height": 960, "label": "1:1 · 960×960 (non-native, experimental)", "native": False},
    {"id": "4x5", "aspect": "4:5", "width": 832, "height": 1024, "label": "4:5 · 832×1024 (non-native, experimental)", "native": False},
    {"id": "16x9-draft", "aspect": "16:9", "width": 832, "height": 480, "label": "16:9 · 832×480 draft (faster, lower quality)", "native": False},
    {"id": "9x16-draft", "aspect": "9:16", "width": 480, "height": 832, "label": "9:16 · 480×832 draft (faster, lower quality)", "native": False},
]

WORKFLOWS = {
    "wan22-i2v": {
        "id": "wan22-i2v",
        "kind": "image-to-video",
        "label": "Image → Video · Wan 2.2 TI2V 5B",
        "model": "Wan 2.2 TI2V 5B",
        "license": "Apache-2.0",
        "requires_image": True,
        "files": WAN22_FILES,
        "nodes": ["UNETLoader", "CLIPLoader", "VAELoader", "ModelSamplingSD3", "Wan22ImageToVideoLatent", "KSampler", "VAEDecode", "CreateVideo", "SaveVideo"],
    },
    "wan22-t2v": {
        "id": "wan22-t2v",
        "kind": "text-to-video",
        "label": "Text → Video · Wan 2.2 TI2V 5B",
        "model": "Wan 2.2 TI2V 5B",
        "license": "Apache-2.0",
        "requires_image": False,
        "files": WAN22_FILES,
        "nodes": ["UNETLoader", "CLIPLoader", "VAELoader", "ModelSamplingSD3", "Wan22ImageToVideoLatent", "KSampler", "VAEDecode", "CreateVideo", "SaveVideo"],
    },
}

WAN22_LIMITS = {
    "sizes": WAN22_SIZES,
    "fps": 24,
    "frames": {"min": 25, "max": 121, "step": 4, "default": 121},  # 4n+1 frames
    "steps": {"min": 8, "max": 50, "default": 20},
    "cfg": {"min": 1.0, "max": 10.0, "default": 5.0},
    "prompt_max": 2000,
    "negative_default": WAN_DEFAULT_NEGATIVE,
    "output": "mp4",
}

SEED_MAX = 2**63 - 1


class ValidationError(ValueError):
    pass


def validate_params(workflow_id: str, raw: dict) -> dict:
    """Checks a job request against what the workflow really supports. Returns clean params."""
    if workflow_id not in WORKFLOWS:
        raise ValidationError("Unknown workflow.")
    lim = WAN22_LIMITS
    prompt = str(raw.get("prompt") or "").strip()
    if not prompt:
        raise ValidationError("Write a prompt describing the motion or scene.")
    if len(prompt) > lim["prompt_max"]:
        raise ValidationError(f"Prompt is too long (max {lim['prompt_max']} characters).")
    negative = str(raw.get("negative") if raw.get("negative") is not None else lim["negative_default"]).strip()[: lim["prompt_max"]]

    size = next((s for s in lim["sizes"] if s["id"] == raw.get("size")), None)
    if not size:
        raise ValidationError("Choose one of the supported sizes.")

    def num(key, kind, rng):
        try:
            v = kind(raw.get(key, rng["default"]))
        except (TypeError, ValueError):
            raise ValidationError(f"{key} must be a number.") from None
        if not (rng["min"] <= v <= rng["max"]):
            raise ValidationError(f"{key} must be between {rng['min']} and {rng['max']}.")
        return v

    frames = num("frames", int, lim["frames"])
    if (frames - 1) % 4:
        raise ValidationError("Frame count must be 4n+1 (e.g. 49, 81, 121).")
    steps = num("steps", int, lim["steps"])
    cfg = round(num("cfg", float, lim["cfg"]), 2)

    seed = raw.get("seed")
    if seed in (None, "", -1, "-1"):
        import secrets
        seed = secrets.randbelow(SEED_MAX)
    try:
        seed = int(seed)
    except (TypeError, ValueError):
        raise ValidationError("Seed must be a whole number.") from None
    if not (0 <= seed <= SEED_MAX):
        raise ValidationError("Seed is out of range.")

    return {
        "prompt": prompt,
        "negative": negative,
        "size": size["id"],
        "width": size["width"],
        "height": size["height"],
        "frames": frames,
        "fps": lim["fps"],
        "steps": steps,
        "cfg": cfg,
        "seed": seed,
        "duration_s": round(frames / lim["fps"], 2),
    }


def build_graph(workflow_id: str, p: dict, image_name: str | None, filename_prefix: str) -> dict:
    """ComfyUI API-format graph, node-for-node the official Wan 2.2 5B TI2V template."""
    f = WORKFLOWS[workflow_id]["files"]
    latent = {"vae": ["39", 0], "width": p["width"], "height": p["height"], "length": p["frames"], "batch_size": 1}
    graph = {
        "37": {"class_type": "UNETLoader", "inputs": {"unet_name": f["diffusion_models"], "weight_dtype": "default"}},
        "38": {"class_type": "CLIPLoader", "inputs": {"clip_name": f["text_encoders"], "type": "wan", "device": "default"}},
        "39": {"class_type": "VAELoader", "inputs": {"vae_name": f["vae"]}},
        "48": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["37", 0], "shift": 8.0}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": p["prompt"], "clip": ["38", 0]}},
        "7": {"class_type": "CLIPTextEncode", "inputs": {"text": p["negative"], "clip": ["38", 0]}},
        "55": {"class_type": "Wan22ImageToVideoLatent", "inputs": latent},
        "3": {"class_type": "KSampler", "inputs": {
            "seed": p["seed"], "steps": p["steps"], "cfg": p["cfg"], "sampler_name": "uni_pc", "scheduler": "simple", "denoise": 1.0,
            "model": ["48", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["55", 0]}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["39", 0]}},
        "57": {"class_type": "CreateVideo", "inputs": {"images": ["8", 0], "fps": float(p["fps"])}},
        # Newer ComfyUI keys the nested codec as "format.codec"; older builds use "codec". Unused keys are ignored.
        "58": {"class_type": "SaveVideo", "inputs": {"video": ["57", 0], "filename_prefix": filename_prefix, "format": "mp4", "codec": "auto", "format.codec": "auto"}},
    }
    if WORKFLOWS[workflow_id]["requires_image"]:
        if not image_name:
            raise ValidationError("Upload a reference image first.")
        graph["56"] = {"class_type": "LoadImage", "inputs": {"image": image_name}}
        latent["start_image"] = ["56", 0]
    return graph


STAGE_BY_NODE = {"3": "Generating frames", "8": "Decoding video", "58": "Saving video", "37": "Loading model", "38": "Loading text encoder", "39": "Loading VAE"}


def hardware_guidance(vram_gb: float | None, device_type: str | None) -> dict:
    """Plain-language fit for Wan 2.2 5B. Based on ComfyUI's note that the 5B model runs on 8 GB with offloading."""
    if vram_gb is None:
        return {"level": "unknown", "text": "GPU not detected yet. Start ComfyUI so the studio can read your hardware."}
    if device_type and device_type != "cuda":
        return {"level": "unsupported", "text": f"Detected a {device_type} device. Version 1 is set up and tested for NVIDIA (CUDA) GPUs only."}
    if vram_gb < 7.5:
        return {"level": "poor", "text": f"{vram_gb:.0f} GB VRAM is below what Wan 2.2 5B needs. Expect out-of-memory errors; use draft sizes and short clips."}
    if vram_gb < 11.5:
        return {"level": "tight", "text": f"{vram_gb:.0f} GB VRAM: should run with ComfyUI's automatic offloading, slowly. Start with draft sizes and 49–81 frames."}
    if vram_gb < 15.5:
        return {"level": "ok", "text": f"{vram_gb:.0f} GB VRAM: native 720p should work; full 121-frame clips will take a while."}
    return {"level": "good", "text": f"{vram_gb:.0f} GB VRAM: comfortable for native 720p clips."}

In [ ]:
import sys, time, urllib.request
sys.path.insert(0, "/tmp")
log = open(f"{WORK}/comfyui.log", "w")
# Bound to 127.0.0.1 only — nothing is exposed to the internet.
server = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"], cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI did not start — see the log: " + open(f"{WORK}/comfyui.log").read()[-3000:])
from comfy import ComfyClient
comfy = ComfyClient("http://127.0.0.1:8188")
comfy.start_progress_listener()
dev = comfy.system_stats()["devices"][0]
print("ComfyUI running on", dev["name"], f"({dev['vram_total']/1024**3:.1f} GB VRAM)")

## 5 · Your reference image and prompt

Set `MODE = "text"` to generate from the prompt alone (no upload needed), or keep `"image"` and upload a photo:
right sidebar → **Input** → **Upload** → *New Dataset* → add your photo (PNG/JPEG/WebP) → **Create**.
It appears under `/kaggle/input/…`. The first image found there is used. Or set `IMAGE_URL` to a public image link.

Start small to prove it works: **draft size, 49 frames (≈2 s)**. Bigger settings take much longer on a T4.

In [ ]:
import glob
MODE = "text"           # "text" = video from the prompt alone · "image" = animate your uploaded image
IMAGE_URL = ""          # optional: direct link to a public image instead of uploading
PROMPT = "A small red sports car driving along a coastal road at sunset, cinematic lighting, natural wheel movement, smooth camera tracking, realistic motion."
SIZE = "auto"           # auto picks 9x16-draft or 16x9-draft from the image shape; or e.g. "9x16", "16x9", "1x1"
FRAMES = 49             # 25, 49, 81 or 121 (24 fps)
STEPS = 20
SEED = 42

if MODE == "text":
    IMAGE = None
elif IMAGE_URL:
    IMAGE = f"{WORK}/reference" + os.path.splitext(IMAGE_URL.split("?")[0])[1].lower()
    urllib.request.urlretrieve(IMAGE_URL, IMAGE)
else:
    found = sorted(p for ext in ("png", "jpg", "jpeg", "webp", "PNG", "JPG", "JPEG", "WEBP") for p in glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True))
    assert found, "No image found. Upload one (see above) or set IMAGE_URL."
    IMAGE = found[0]
from PIL import Image
w, h = Image.open(IMAGE).size if IMAGE else (16, 9)
if SIZE == "auto":
    SIZE = "9x16-draft" if h > w else "16x9-draft"
print("Mode:", MODE, "| image:", IMAGE, "→ size", SIZE)
Image.open(IMAGE).resize((w * 240 // max(w, h), h * 240 // max(w, h))) if IMAGE else None

## 6 · Generate (real progress from ComfyUI)

In [ ]:
from workflows import build_graph, validate_params, STAGE_BY_NODE
WORKFLOW = "wan22-t2v" if MODE == "text" else "wan22-i2v"
params = validate_params(WORKFLOW, {"prompt": PROMPT, "size": SIZE, "frames": FRAMES, "steps": STEPS, "seed": SEED})
image_name = None
if IMAGE:
    mime = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".webp": "image/webp"}[os.path.splitext(IMAGE)[1].lower()]
    image_name = comfy.upload_image("reference" + os.path.splitext(IMAGE)[1].lower(), open(IMAGE, "rb").read(), mime)
prompt_id = comfy.queue_prompt(build_graph(WORKFLOW, params, image_name, "studio/kaggle_first"))
print("Queued", prompt_id, params)

t0, last = time.time(), ""
while True:
    hist = comfy.history(prompt_id)
    if hist:
        break
    p = comfy.progress_for(prompt_id)
    line = f"{STAGE_BY_NODE.get(str(p['node']), 'Working')} {p['value']}/{p['max']}" if p else "Loading models (first time takes a few minutes)"
    if line != last:
        print(f"[{int(time.time()-t0):>4}s] {line}"); last = line
    time.sleep(3)

status = hist.get("status", {})
if status.get("status_str") == "error":
    msgs = [m[1].get("exception_message") for m in status.get("messages", []) if m[0] == "execution_error"]
    raise RuntimeError(f"Generation failed: {msgs} — last log lines:\n" + open(f"{WORK}/comfyui.log").read()[-2000:])
out = next(i for node in hist["outputs"].values() for items in node.values() if isinstance(items, list) for i in items if isinstance(i, dict) and str(i.get("filename", "")).endswith(".mp4"))
RESULT = "/kaggle/working/first_video.mp4"
comfy.download_output(out["filename"], out.get("subfolder", ""), out.get("type", "output"), RESULT)
ELAPSED = int(time.time() - t0)
print(f"Done in {ELAPSED}s →", RESULT, f"({os.path.getsize(RESULT)/1e6:.1f} MB)")

## 7 · Check, watch and download

In [ ]:
print(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "stream=codec_name,width,height,nb_frames:format=duration", "-of", "default=nw=1", RESULT], capture_output=True, text=True).stdout or "(ffprobe not available)")
from IPython.display import Video, FileLink, display
display(Video(RESULT, embed=True, width=360))
display(FileLink("first_video.mp4"))  # also in the right sidebar → Output → first_video.mp4

## 8 · Gallery entry (for publishing this clip on blackdisk.co/studio)

In [ ]:
import json, datetime
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.splitlines()[0].strip()
size = next(s for s in __import__("workflows").WAN22_SIZES if s["id"] == params["size"])
print(json.dumps({
    "id": f"clip-{datetime.date.today():%Y%m%d}-{params['seed']}",
    "title": "",
    "date": f"{datetime.date.today()}",
    "video": "/studio/videos/REPLACE.mp4",
    "mode": "text-to-video" if WORKFLOW == "wan22-t2v" else "image-to-video",
    "prompt": params["prompt"],
    "width": size["width"], "height": size["height"],
    "frames": params["frames"], "fps": 24, "steps": params["steps"], "cfg": params["cfg"], "seed": params["seed"],
    "gpu": gpu, "seconds": ELAPSED,
}, indent=2, ensure_ascii=False))

### Then
- Download `first_video.mp4` (right sidebar → **Output**) and keep the gallery entry above with it.
- Stop the session (power icon) so you don't use up free GPU hours.
- Try more: change `PROMPT`/`SEED`, `FRAMES = 81`, or `SIZE = "9x16"` (native 720p, much slower) and re-run steps 5–8 only.